# 05 QC Threshold Calibration

Run this after D3 has produced at least 50 generated samples. It computes mean/p99 offset distributions and estimates thresholds with a 60-85% target pass rate.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np

from blast_pile_diffusion.qc.qc_runner import run_qc_batch

GENERATED_DIR = Path('../data/generated')
PREPROCESSED_DIR = Path('../data/preprocessed')
QC_CONFIG = Path('../configs/qc/thresholds.yaml')
OUTPUT_DIR = Path('../notebooks/outputs/qc_threshold_calibration')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report = run_qc_batch(
    GENERATED_DIR,
    PREPROCESSED_DIR,
    QC_CONFIG,
    save_debug_images=False,
    report_path=OUTPUT_DIR / 'qc_calibration_report.json',
)
records = []
for sample_dir in GENERATED_DIR.iterdir():
    qc_path = sample_dir / 'qc.json'
    if qc_path.exists():
        records.append(json.loads(qc_path.read_text(encoding='utf-8')))
assert len(records) >= 50, f'E2 expects at least 50 generated samples, got {len(records)}.'


In [ ]:
mean_offsets = np.array([r['mean_offset_px'] for r in records if r.get('mean_offset_px') is not None], dtype=float)
p99_offsets = np.array([r['p99_offset_px'] for r in records if r.get('p99_offset_px') is not None], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(mean_offsets, bins=40)
axes[0].set_title('mean_offset_px')
axes[1].hist(p99_offsets, bins=40)
axes[1].set_title('p99_offset_px')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'offset_histograms.png', dpi=160)
plt.show()

threshold_candidates = np.percentile(mean_offsets, [60, 70, 80, 85])
candidate_table = []
for threshold in threshold_candidates:
    pass_rate = float((mean_offsets <= threshold).mean() * 100)
    candidate_table.append({'mean_threshold': float(threshold), 'pass_rate_pct': pass_rate})
candidate_table


In [ ]:
borderline_count = 10
chosen = candidate_table[1]['mean_threshold']
borderline = sorted(records, key=lambda r: abs((r.get('mean_offset_px') or 1e9) - chosen))[:borderline_count]
(OUTPUT_DIR / 'threshold_candidates.json').write_text(json.dumps(candidate_table, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'borderline_samples.json').write_text(json.dumps(borderline, indent=2), encoding='utf-8')
print('Review borderline samples before writing final thresholds to configs/qc/thresholds.yaml')
candidate_table
